# 00 · Entorno y configuración — American Tower Perú
## Propósito y forma de ejecución
Ejecute de arriba abajo en **EMR Studio conectado al clúster, kernel PySpark (Livy)**, o en Jupyter sobre EMR con sesión Spark existente. Las celdas usan Python ordinario compatible con PySpark; no usan `%pip` ni `%%local`. El proceso Python remoto puede estar en un contenedor YARN: su carpeta no es la carpeta del navegador ni necesariamente `/home/hadoop` del nodo primario.

El clúster declarado tiene un nodo primario y tres trabajadores m5.2xlarge. El primario de EMR y el driver de una aplicación Spark son conceptos distintos. Esta notebook mide el entorno observado; no supone tres procesos executor ni reserva toda la RAM. Ingesta Bronze requiere AWS/S3; Sedona se diagnostica para la fase Silver.

**Escrituras:** configuración en Artifacts; evidencia y prueba pequeña en Operations. No se consulta ni modifica el bucket Terraform State. No se cambian políticas IAM, red o configuración del clúster.

**Versión alineada al contrato 1.0.0.** Sentinel nacional 2019Q1–2026Q1, selección mensual por tesela. CONFIG incorpora todo el contrato, sin decidir inputs de modelos en esta notebook. Instale requirements.txt en el Python remoto antes de continuar. No instala/reemplaza PySpark de EMR.

## 0. Creación de sesión Spark y verificación de entorno
Cree una sesión Spark y verifique que no sea local.

In [1]:
import os
import sys
from pathlib import Path
from importlib.metadata import version

PYTHON_ENV = "/opt/american-tower/venv/bin/python"
SPARK_HOME = Path("/usr/lib/spark")

assert sys.executable == PYTHON_ENV, (
    f"Selecciona el kernel {PYTHON_ENV}. Actual: {sys.executable}"
)
assert version("apache-sedona") == "1.7.2"

# Seleccionar Python antes de crear cualquier SparkContext.
os.environ["SPARK_HOME"] = str(SPARK_HOME)
os.environ["PYSPARK_PYTHON"] = PYTHON_ENV
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_ENV

py4j = sorted(
    (SPARK_HOME / "python" / "lib").glob("py4j-*.zip")
)
if not py4j:
    raise RuntimeError("No se encontró Py4J en la instalación de EMR.")

for path in [SPARK_HOME / "python", py4j[0]]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from pyspark import SparkContext
from sedona.spark import SedonaContext

if SparkContext._active_spark_context is not None:
    raise RuntimeError(
        "Ya existe un SparkContext. Reinicia el kernel "
        "y ejecuta esta celda antes de cualquier otra inicialización."
    )

spark = (
    SedonaContext.builder()
    .appName("american-tower-entorno")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.pyspark.python", PYTHON_ENV)
    .config("spark.pyspark.driver.python", PYTHON_ENV)
    .config("spark.executorEnv.PYSPARK_PYTHON", PYTHON_ENV)
    .config(
        "spark.jars.packages",
        "org.apache.sedona:sedona-spark-shaded-3.5_2.12:1.7.2,"
        "org.datasyslab:geotools-wrapper:1.7.2-28.5"
    )
    .getOrCreate()
)

sc = spark.sparkContext

print("Python del driver:", sys.executable)
print("Python para tareas:", sc.pythonExec)
print("Master:", sc.master)
print("Aplicación:", sc.applicationId)

assert sc.master == "yarn"
assert sc.pythonExec == PYTHON_ENV, sc.pythonExec

# Si falla, conserva el traceback completo.
spark = SedonaContext.create(spark)

spark.sql(
    "SELECT ST_AsText(ST_Point(-77.04, -12.05)) AS geometria"
).show(truncate=False)

Oct 10, 2026 3:55:19 AM org.apache.spark.launcher.Log4jHotPatchOption staticJavaAgentOption



:: loading settings :: url = jar:file:/usr/lib/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hadoop/.ivy2/cache
The jars for the packages stored in: /home/hadoop/.ivy2/jars
org.apache.sedona#sedona-spark-shaded-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5d057dea-4542-4b14-bb88-7a61c4b7b478;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-shaded-3.5_2.12;1.7.2 in central
	found org.datasyslab#geotools-wrapper;1.7.2-28.5 in central
downloading https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-3.5_2.12/1.7.2/sedona-spark-shaded-3.5_2.12-1.7.2.jar ...
	[SUCCESSFUL ] org.apache.sedona#sedona-spark-shaded-3.5_2.12;1.7.2!sedona-spark-shaded-3.5_2.12.jar (664ms)
downloading https://repo1.maven.org/maven2/org/datasyslab/geotools-wrapper/1.7.2-28.5/geotools-wrapper-1.7.2-28.5.jar ...
	[SUCCESSFUL ] org.datasyslab#geotools-wrapper;1.7.2-28.5!geotools-wrapper.jar (329ms)
:: resolution report :: resolve 1242ms :: artifacts

Python del driver: /opt/american-tower/venv/bin/python
Python para tareas: /opt/american-tower/venv/bin/python
Master: yarn
Aplicación: application_1791597468580_0006


+---------------------+
|geometria            |
+---------------------+
|POINT (-77.04 -12.05)|
+---------------------+



In [2]:
def comprobar_entorno(iterator):
    import sys
    import socket
    import numpy
    import pyarrow
    import shapely
    import geopandas
    import rasterio
    from importlib.metadata import version

    yield {
        "host": socket.gethostname(),
        "python": sys.executable,
        "numpy": numpy.__version__,
        "pyarrow": pyarrow.__version__,
        "sedona": version("apache-sedona"),
    }

resultados = sc.parallelize(
    range(12), 12
).mapPartitions(comprobar_entorno).collect()

for host in sorted({r["host"] for r in resultados}):
    print(next(r for r in resultados if r["host"] == host))

assert all(
    r["python"] == PYTHON_ENV
    and r["numpy"] == "1.26.4"
    and r["pyarrow"] == "17.0.0"
    and r["sedona"] == "1.7.2"
    for r in resultados
)

{'host': 'ip-10-0-11-126.ec2.internal', 'python': '/opt/american-tower/venv/bin/python', 'numpy': '1.26.4', 'pyarrow': '17.0.0', 'sedona': '1.7.2'}
{'host': 'ip-10-0-147-125.ec2.internal', 'python': '/opt/american-tower/venv/bin/python', 'numpy': '1.26.4', 'pyarrow': '17.0.0', 'sedona': '1.7.2'}
{'host': 'ip-10-0-93-251.ec2.internal', 'python': '/opt/american-tower/venv/bin/python', 'numpy': '1.26.4', 'pyarrow': '17.0.0', 'sedona': '1.7.2'}


## 1. Parámetros
Complete `cluster_id` si desea verificar EMR por API. Se conserva la región observada de los buckets; no se inventa una región.

In [3]:
CONFIG = {'schema_version': 2, 'buckets': {'artifacts': 'american-tower-dev-artifacts-74d32893263161eaef72faf57a', 'bronze': 'american-tower-dev-bronze-b7f9699786f4c119558b9d30f9', 'gold': 'american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee', 'operations': 'american-tower-dev-operations-1dd72256126674a5893dfca3df', 'silver': 'american-tower-dev-silver-2ad71fded155d2bd52df8f836e'}, 'cluster_id': 'j-05035651U7GHZ3X2K30B', 'expected_primary_instance': 'm5.2xlarge', 'expected_worker_instances': 3, 'config_key': 'config/emr_ingesta/v1_alineada/project_config.json', 'bronze_root': 'sources', 'evidence_root': 'evidence', 'spark_probe_root': 'diagnostics', 'ingestion_scope': 'Peru_nacional; historia 2019Q1-2026Q1; Sentinel mensual por MGRS; derivados en Operations', 'spec_version': '1.0.0', 'spec': {'spec_version': '1.0.0', 'state': 'implementation_specification_not_executed', 'scope': 'Peru_nacional', 'platform': 'AWS_EMR_S3_Spark_Sedona', 'geography': {'catalog_publisher': 'INEI', 'catalog_announced_version': '2023', 'expected_districts': 1890, 'district_key': 'ubigeo', 'district_key_length': 6}, 'periods': ['2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1'], 'source_policy': {'ookla': {'layer': 'mobile', 'bronze_extent': 'global_originals', 'model_base': True}, 'osiptel_cobertura': {'observed_period': '202303', 'model_base': False}, 'osiptel_estaciones': {'observed_period': None, 'unknown_period_action': 'ingest_snapshot_exclude_from_historical_features', 'model_base': False}, 'worldpop': {'reference_year': 2020, 'model_base': False}, 'sentinel': {'national': True, 'periods': ['2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1'], 'bands': ['red', 'nir', 'swir16', 'scl'], 'selection': 'least_cloud_per_mgrs_tile_per_calendar_month', 'max_scenes_per_tile_per_quarter': 3, 'max_scene_cloud_pct': 100, 'required_footprint_fraction': 0.999999, 'scl_excluded': [0, 1, 3, 8, 9, 10, 11], 'output_grid_m': 20, 'temporal_composite': 'median_of_valid_observations_per_pixel', 'district_valid_fraction_threshold': 0.8, 'sensitivity_thresholds': [0.6, 0.8, 0.9], 'model_base': False}}, 'storage_policy': {'originals_and_provider_metadata': 'bronze', 'derived_aoi_selection_reports_logs': 'operations', 'configuration_and_code': 'artifacts', 'curated': 'silver', 'integrated': 'gold'}, 'prediction': {'unit': 'district_quarter', 'target': 'test_weighted_download_mbps_t_plus_1', 'features': ['download_mbps_t', 'upload_mbps_t', 'latency_ms_t', 'log1p_tests_t', 'tile_count_t', 'quarter_sin_t', 'quarter_cos_t'], 'require_consecutive_quarters': True, 'split_by': 'target_quarter', 'train': ['2019Q2', '2023Q4'], 'validation': ['2024Q1', '2024Q4'], 'test': ['2025Q1', '2026Q1'], 'baseline': 'persistence_download_t', 'candidate': {'algorithm': 'Spark_ML_RandomForestRegressor', 'seed': 42, 'numTrees': 100, 'maxDepth': 8}, 'metrics': ['MAE', 'RMSE', 'R2'], 'evaluation_type': 'retrospective_snapshot_backtest'}, 'operations_to_supply': {'cluster_id': None, 'region': None, 'emr_release': None, 'osiptel_cobertura_uri': None, 'osiptel_estaciones_uri': None, 'max_planned_bytes': None}, 'notebook_alignment_required': False, 'alignment_delivery': '00_01_requirements_contract_v1'}}
PERU_AOI_INPUT_AVAILABLE = False  # True si aportará GeoJSON INEI y no necesita extraer RAR

## 2. SDK, identidad y utilidades
Se utiliza la cadena de credenciales temporales de AWS. No incluya access keys en la notebook.

In [4]:
import json, os, sys, platform, socket, hashlib, time, uuid, logging, shutil, re
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

def utc(): return datetime.now(timezone.utc).isoformat()
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
AWS = boto3.Session()
SDK_CONFIG = Config(retries={'max_attempts': 8, 'mode': 'standard'}, connect_timeout=15, read_timeout=120)
s3 = AWS.client('s3', config=SDK_CONFIG)
def put_json(bucket, key, value):
    return s3.put_object(Bucket=bucket, Key=key, Body=json.dumps(value, ensure_ascii=False, indent=2, default=str).encode(), ContentType='application/json')
def get_json(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    with response['Body'] as body: return json.loads(body.read())
def digest_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
def digest_object(bucket,key,version=None):
    args={'Bucket':bucket,'Key':key}
    if version: args['VersionId']=version
    response=s3.get_object(**args); h=hashlib.sha256(); count=0
    with response['Body'] as body:
        for block in body.iter_chunks(chunk_size=8*1024*1024): h.update(block); count+=len(block)
    return h.hexdigest(),count

In [5]:
identity=AWS.client('sts',config=SDK_CONFIG).get_caller_identity()
REPORT={'run_id':RUN_ID,'started_at':utc(),'identity':identity,'python':sys.version,'host':socket.gethostname(),'platform':platform.platform(),'bucket_checks':[],'checks':[]}
print(json.dumps({'run_id':RUN_ID,'account':identity['Account'],'arn':identity['Arn'],'host':REPORT['host']},indent=2))

{
  "run_id": "20261010T035605Z-8479b3e9",
  "account": "715010567123",
  "arn": "arn:aws:sts::715010567123:assumed-role/american-tower-dev-emr-ec2-role/i-0ae1f4ea8e07fda5e",
  "host": "ip-10-0-75-90.ec2.internal"
}


## 3. Buckets y controles observables
`HeadBucket` es obligatorio en los cinco buckets. Las lecturas de cifrado, versionado y bloqueo público pueden requerir permisos administrativos adicionales: se registran como no verificadas cuando son denegadas. No equivalen a una auditoría completa de seguridad.

In [6]:
for zone,bucket in CONFIG['buckets'].items():
    head=s3.head_bucket(Bucket=bucket)
    row={'zone':zone,'bucket':bucket,'region':head.get('ResponseMetadata',{}).get('HTTPHeaders',{}).get('x-amz-bucket-region')}
    for name,method in [('encryption',s3.get_bucket_encryption),('versioning',s3.get_bucket_versioning),('public_access_block',s3.get_public_access_block)]:
        try: row[name]=method(Bucket=bucket)
        except ClientError as exc: row[name]={'verified':False,'code':exc.response['Error']['Code']}
    REPORT['bucket_checks'].append(row)
    print(zone,bucket,row['region'])
regions={r['region'] for r in REPORT['bucket_checks'] if r['region']}
CONFIG['observed_bucket_regions']=sorted(regions)
CONFIG['account_id']=identity['Account']

artifacts american-tower-dev-artifacts-74d32893263161eaef72faf57a us-east-1
bronze american-tower-dev-bronze-b7f9699786f4c119558b9d30f9 us-east-1
gold american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee us-east-1
operations american-tower-dev-operations-1dd72256126674a5893dfca3df us-east-1
silver american-tower-dev-silver-2ad71fded155d2bd52df8f836e us-east-1


## 4. Clúster observado (opcional)
Con `cluster_id=None` esta comprobación queda pendiente. `ListInstances` no demuestra por sí solo el número de executors usados por la sesión Spark.

In [7]:
if CONFIG['cluster_id']:
    region=next(r['region'] for r in REPORT['bucket_checks'] if r['zone']=='operations')
    emr=AWS.client('emr',region_name=region,config=SDK_CONFIG)
    try:
        cluster=emr.describe_cluster(ClusterId=CONFIG['cluster_id'])['Cluster']
        instances=[]
        for page in emr.get_paginator('list_instances').paginate(ClusterId=CONFIG['cluster_id']): instances.extend(page['Instances'])
        REPORT['emr']={'id':cluster['Id'],'release':cluster.get('ReleaseLabel'),'status':cluster['Status'],'applications':cluster.get('Applications'),'instances':instances}
        print(json.dumps(REPORT['emr'],default=str,indent=2))
    except ClientError as exc: REPORT['emr']={'verified':False,'code':exc.response['Error']['Code']}; print(REPORT['emr'])
else:
    REPORT['emr']={'verified':False,'reason':'cluster_id no configurado'}
    print(REPORT['emr'])

{
  "id": "j-05035651U7GHZ3X2K30B",
  "release": "emr-7.1.0",
  "status": {
    "State": "WAITING",
    "StateChangeReason": {
      "Message": "Cluster ready to run steps."
    },
    "Timeline": {
      "CreationDateTime": "2026-10-10 01:54:32.858000+00:00",
      "ReadyDateTime": "2026-10-10 01:59:19.160000+00:00"
    }
  },
  "applications": [
    {
      "Name": "Hadoop",
      "Version": "3.3.6"
    },
    {
      "Name": "Spark",
      "Version": "3.5.0"
    }
  ],
  "instances": [
    {
      "Id": "ci-03287022S8LD1Z0G4EPB",
      "Ec2InstanceId": "i-069be0e960ac4a2da",
      "PublicDnsName": "ec2-75-101-240-54.compute-1.amazonaws.com",
      "PublicIpAddress": "75.101.240.54",
      "PrivateDnsName": "ip-10-0-11-126.ec2.internal",
      "PrivateIpAddress": "10.0.11.126",
      "Status": {
        "State": "RUNNING",
        "StateChangeReason": {},
        "Timeline": {
          "CreationDateTime": "2026-10-10 01:54:40.435000+00:00",
          "ReadyDateTime": "2026-10-10 01:

## 5. Spark y ejecutores
La variable `spark` debe provenir del kernel conectado a EMR. Se rechaza `local[*]` para evitar una validación local accidental. Las muestras de hosts solo reflejan tareas observadas.

In [8]:
if 'spark' not in globals():
    raise RuntimeError('Seleccione kernel PySpark conectado a EMR; falta la sesión spark.')
sc=spark.sparkContext
if sc.master.startswith('local'): raise RuntimeError('Se detectó Spark local, no el clúster EMR.')
settings={k:sc.getConf().get(k,'no_configurado') for k in ['spark.executor.instances','spark.executor.cores','spark.executor.memory','spark.driver.memory','spark.dynamicAllocation.enabled','spark.eventLog.enabled','spark.eventLog.dir','spark.sql.adaptive.enabled']}
def worker_probe(iterator):
    import socket,sys
    count=sum(1 for _ in iterator)
    yield {'host':socket.gethostname(),'python':sys.version.split()[0],'records':count}
workers=sc.parallelize(range(96),12).mapPartitions(worker_probe).collect()
assert sum(x['records'] for x in workers)==96
REPORT['spark']={'version':spark.version,'master':sc.master,'application_id':sc.applicationId,'default_parallelism':sc.defaultParallelism,'settings':settings,'worker_samples':workers}
print(json.dumps(REPORT['spark'],indent=2))

{
  "version": "3.5.0-amzn-1",
  "master": "yarn",
  "application_id": "application_1791597468580_0006",
  "default_parallelism": 12,
  "settings": {
    "spark.executor.instances": "no_configurado",
    "spark.executor.cores": "4",
    "spark.executor.memory": "9486M",
    "spark.driver.memory": "2048M",
    "spark.dynamicAllocation.enabled": "true",
    "spark.eventLog.enabled": "true",
    "spark.eventLog.dir": "hdfs:///var/log/spark/apps",
    "spark.sql.adaptive.enabled": "no_configurado"
  },
  "worker_samples": [
    {
      "host": "ip-10-0-147-125.ec2.internal",
      "python": "3.9.25",
      "records": 8
    },
    {
      "host": "ip-10-0-93-251.ec2.internal",
      "python": "3.9.25",
      "records": 8
    },
    {
      "host": "ip-10-0-11-126.ec2.internal",
      "python": "3.9.25",
      "records": 8
    },
    {
      "host": "ip-10-0-147-125.ec2.internal",
      "python": "3.9.25",
      "records": 8
    },
    {
      "host": "ip-10-0-93-251.ec2.internal",
      "py

## 6. Dependencias remotas, sistema y Sedona
Requiere imports reales de las dependencias de ingesta y extractor RAR (salvo AOI aportado). Muestrea versiones en executors para Silver. Sedona pendiente no bloquea transferencias Bronze, pero debe quedar listo antes de operaciones espaciales Spark. Registra todas las versiones instaladas para congelar el entorno observado.

In [9]:
from importlib.metadata import version,PackageNotFoundError,distributions
REPORT['packages']={}
required=['boto3','requests','numpy','pandas','shapely','pyproj','pyogrio','geopandas','apache-sedona']
missing=[]
for package in required+['rasterio','pyarrow','openpyxl']:
    try: REPORT['packages'][package]=version(package)
    except PackageNotFoundError:
        REPORT['packages'][package]='no_instalado'
        if package in required: missing.append(package)
REPORT['python_executable']=sys.executable
REPORT['installed_distributions']={d.metadata['Name']:d.version for d in distributions() if d.metadata['Name']}
REPORT['rar_extractor']=shutil.which('unrar') or shutil.which('bsdtar')
# Ensayo de imports reales: tener metadata de una distribución no prueba su carga binaria.
try:
    import requests,numpy,pandas,shapely,pyproj,pyogrio,geopandas
    REPORT['geospatial_imports_ok']=True
except Exception as exc:
    REPORT['geospatial_imports_ok']=False
    REPORT['geospatial_import_error']=type(exc).__name__
if not ((3,9)<=sys.version_info[:2]<(3,12)):
    raise RuntimeError('Perfil requirements soportado: Python 3.9–3.11. Revisar entorno antes de instalar.')
if missing or not REPORT['geospatial_imports_ok']:
    raise RuntimeError('Dependencias remotas faltantes o no importables: '+str(missing)+'. Instale requirements.txt en el Python del proceso remoto.')
if not REPORT['rar_extractor'] and not PERU_AOI_INPUT_AVAILABLE:
    raise RuntimeError('Falta unrar/bsdtar de sistema. Instalar o aportar GeoJSON INEI y configurar PERU_AOI_INPUT_AVAILABLE.')
try:
    from sedona.spark import SedonaContext
    if spark.version.split('.')[:2]!=['3','5']:
        raise RuntimeError('Perfil JAR Sedona entregado requiere Spark 3.5 / Scala 2.12; adaptar perfil para otra release.')
    SedonaContext.create(spark)
    REPORT['sedona']={'ok':True,'sample':spark.sql("SELECT ST_AsText(ST_Point(-79.0,-8.1)) AS point").first()['point']}
except Exception as exc:
    REPORT['sedona']={'ok':False,'error_type':type(exc).__name__,'action':'Ver INSTALACION_DEPENDENCIAS.md; preparar JAR antes de iniciar sesión. Bronze no usa Sedona en transferencias.'}
def executor_dependencies(iterator):
    from importlib.metadata import version
    import socket,sys
    packages={}
    for name in ['apache-sedona','shapely','numpy','geopandas','pyproj','pyogrio','rasterio','pyarrow']:
        try: packages[name]=version(name)
        except Exception: packages[name]='no_instalado'
    yield {'host':socket.gethostname(),'python':sys.version.split()[0],'packages':packages}
REPORT['executor_dependency_samples']=sc.parallelize(range(12),12).mapPartitions(executor_dependencies).collect()
print(json.dumps({'python_executable':REPORT['python_executable'],'packages':REPORT['packages'],'rar_extractor':REPORT['rar_extractor'],'sedona':REPORT['sedona'],'executor_dependency_samples':REPORT['executor_dependency_samples']},indent=2))

26/10/10 03:56:19 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
26/10/10 03:56:19 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
26/10/10 03:56:19 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
26/10/10 03:56:19 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.geometryObjects.Geography, which is already registered.
26/10/10 03:56:19 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
26/10/10 03:56:19 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
26/10/10 03:56:19 WARN SimpleFunctionRegistry: The function st_intersection_aggr replaced a previously registered function.
26/10/10 03:56:19 WARN SimpleFunctionRegistry: The function st_union_aggr replaced a previously

{
  "python_executable": "/opt/american-tower/venv/bin/python",
  "packages": {
    "boto3": "1.42.97",
    "requests": "2.32.5",
    "numpy": "1.26.4",
    "pandas": "2.2.3",
    "shapely": "2.0.6",
    "pyproj": "3.6.1",
    "pyogrio": "0.10.0",
    "geopandas": "1.0.1",
    "apache-sedona": "1.7.2",
    "rasterio": "1.3.11",
    "pyarrow": "17.0.0",
    "openpyxl": "3.1.5"
  },
  "rar_extractor": "/usr/bin/bsdtar",
  "sedona": {
    "ok": true,
    "sample": "POINT (-79 -8.1)"
  },
  "executor_dependency_samples": [
    {
      "host": "ip-10-0-11-126.ec2.internal",
      "python": "3.9.25",
      "packages": {
        "apache-sedona": "1.7.2",
        "shapely": "2.0.6",
        "numpy": "1.26.4",
        "geopandas": "1.0.1",
        "pyproj": "3.6.1",
        "pyogrio": "0.10.0",
        "rasterio": "1.3.11",
        "pyarrow": "17.0.0"
      }
    },
    {
      "host": "ip-10-0-147-125.ec2.internal",
      "python": "3.9.25",
      "packages": {
        "apache-sedona": "1.7.2"

## 7. Prueba de lectura/escritura distribuida S3
Escribe 12 filas Parquet en Operations con ruta única y las lee con Spark. No borra objetos; el prefijo diagnostics puede tener una regla de retención definida por Terraform.

In [10]:
probe=f"s3://{CONFIG['buckets']['operations']}/{CONFIG['spark_probe_root']}/run_id={RUN_ID}/spark_probe"
spark.range(12).repartition(3).write.mode('errorifexists').parquet(probe)
check=spark.read.parquet(probe)
assert check.count()==12 and check.agg({'id':'sum'}).first()[0]==66
REPORT['spark_s3_probe']={'ok':True,'uri':probe,'rows':12}
print(REPORT['spark_s3_probe'])

{'ok': True, 'uri': 's3://american-tower-dev-operations-1dd72256126674a5893dfca3df/diagnostics/run_id=20261010T035605Z-8479b3e9/spark_probe', 'rows': 12}


## 8. Publicar configuración y evidencia
Solo se publica después de las pruebas obligatorias. La notebook 01 cargará esta configuración por S3 en una sesión independiente. Registre las rutas impresas.

In [11]:
CONFIG['environment_run_id']=RUN_ID
CONFIG['published_at']=utc()
CONFIG['environment_python']=REPORT['python_executable']
CONFIG['environment_package_versions']=REPORT['packages']
REPORT['finished_at']=utc()
evidence=f"{CONFIG['evidence_root']}/environment/run_id={RUN_ID}/environment.json"
put_json(CONFIG['buckets']['operations'],evidence,REPORT)
put_json(CONFIG['buckets']['artifacts'],f'config/emr_ingesta/runs/{RUN_ID}.json',CONFIG)
put_json(CONFIG['buckets']['artifacts'],CONFIG['config_key'],CONFIG)
print('CONFIG_URI =',f"s3://{CONFIG['buckets']['artifacts']}/{CONFIG['config_key']}")
print('ENVIRONMENT_REPORT =',f"s3://{CONFIG['buckets']['operations']}/{evidence}")
print('ENTORNO_BRONZE_LISTO; Sedona listo:',REPORT['sedona']['ok'])

CONFIG_URI = s3://american-tower-dev-artifacts-74d32893263161eaef72faf57a/config/emr_ingesta/v1_alineada/project_config.json
ENVIRONMENT_REPORT = s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/evidence/environment/run_id=20261010T035605Z-8479b3e9/environment.json
ENTORNO_BRONZE_LISTO; Sedona listo: True
